# Chapter 48: Deployment, Drift, Monitoring, and Retraining

Synthetic NRG production batches demonstrate service indicators, distribution drift, missingness, and alert states.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.monitoring import population_stability_index,latency_summary,missing_rate,breach
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(48);reference=rng.normal(60,12,1200);current=rng.normal(67,14,800);latency=rng.lognormal(3.7,.35,800);distance=current.copy();distance[rng.choice(800,32,replace=False)]=np.nan
print(f'Current records={len(current)}; missing distance={missing_rate(distance):.3f}')


Current records=800; missing distance=0.040


In [ ]:
psi=population_stability_index(reference,current,10);service=latency_summary(latency);print(f'PSI={psi:.3f}; drift state={breach(psi,.10,.25)}');print(f"Latency p50={service['p50']:.1f} ms; p95={service['p95']:.1f} ms; p99={service['p99']:.1f} ms")


PSI=0.256; drift state=critical
Latency p50=40.9 ms; p95=72.7 ms; p99=87.7 ms


In [ ]:
weeks=np.arange(1,9);psi_history=np.array([.03,.04,.06,.08,.12,.17,.22,psi]);states=[breach(v,.10,.25) for v in psi_history];print('Weekly states:',states)


Weekly states: ['ok', 'ok', 'ok', 'ok', 'warning', 'warning', 'warning', 'critical']


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4));axes[0].hist(reference,bins=25,alpha=.6,density=True,label='reference');axes[0].hist(current,bins=25,alpha=.6,density=True,label='current');axes[0].set(title='Distance distribution',xlabel='Route distance');axes[0].legend();axes[1].plot(weeks,psi_history,'o-');axes[1].axhline(.1,color='orange',ls=':');axes[1].axhline(.25,color='red',ls=':');axes[1].set(title='Drift signal by week',xlabel='Week',ylabel='PSI');fig.tight_layout();plt.show()


## Interpretation

The drift threshold opens an investigation; it does not order automatic retraining. The team must check units, route mix, seasonality, model performance after labels mature, and business impact before choosing rollback, recalibration, feature repair, or retraining.


In [ ]:
# Practice: define warning and critical thresholds for missingness and latency.
